# Cross-Screen Measurement: Conversion Drivers (PySpark)

**What this is.** A self-contained measurement-science workflow
(SQL/Python/Databricks/Spark). It joins three disparate sources on a
household-ID spine, cleans them, reduces them to decision-grade metrics, and
runs a weighted drivers analysis: the standard shape of cross-screen panel
measurement work.

**The data model is industry-generic:** ad exposure files from a publisher,
conversion data from a measurement partner, and panel weights from a household
measurement panel.

| Source | Stands in for |
|---|---|
| `ad_exposure.csv` | ad exposure files from a publisher |
| `conversions.csv` | conversion data from a measurement partner |
| `panel.csv` | demographics + panel weights from the measurement panel |

**Six operations, each isolated below:** Retrieve, Link, Shape, Reduce, Infer, Translate.

**How to run**
- *Databricks Community Edition (recommended):* upload this notebook, upload the
  three CSVs to DBFS, set `DATA` to that path, attach to the free cluster, Run All.
- *Locally:* `pip install pyspark` (needs Java 11+), keep `DATA = "data"`, then run.
  If the CSVs are absent the first cell regenerates them deterministically.

## Config

One block governs the run. Changing the campaign key or the model formula here,
rather than editing logic downstream, is what turns a one-off pull into a
*repeatable process* (a template rather than a script).

In [ ]:
# --- Config: the only block you edit to re-point this workflow ---
DATA = "data"                       # local path; in Databricks use e.g. "/FileStore/measurement"
CAMPAIGN_ID = "CMP_2026_Q2_AUTO"    # scope filter, if you add more campaigns
MODEL_FORMULA = "converted ~ exposure_frequency + viewing_profile + segment"
WEIGHT_COL = "panel_weight"         # panel weights make estimates population-representative

# Bootstrap sample data locally if missing (deterministic, seed=42). No-op on Databricks.
import os, subprocess, sys
if DATA == "data" and not os.path.exists(os.path.join(DATA, "panel.csv")):
    subprocess.run([sys.executable, "generate_sample_data.py"], check=True)

## Spark session

Works unchanged in two environments. On Databricks a `spark` session already
exists, so we reuse it; locally we build one in `local[*]` mode. The `try/except`
is the portability seam, not boilerplate.

In [ ]:
try:
    spark  # Databricks provides this
except NameError:
    from pyspark.sql import SparkSession
    spark = (SparkSession.builder
             .appName("panel-measurement-demo")
             .master("local[*]")
             .config("spark.driver.host", "localhost")
             .config("spark.sql.shuffle.partitions", "4")
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")

from pyspark.sql import functions as F, Window
print("Spark", spark.version, "ready")

## 1. Retrieve

**The Why.** Three systems, three files, three owners. Nothing useful happens
until the data is in one engine. `inferSchema` lets Spark type the columns so
`exposure_frequency` arrives as an integer, not text, which matters the moment
we do arithmetic on it.

**The How.** Read each CSV into a DataFrame and confirm row counts before trusting anything.

In [ ]:
exposure = spark.read.csv(f"{DATA}/ad_exposure.csv", header=True, inferSchema=True)
conv     = spark.read.csv(f"{DATA}/conversions.csv",  header=True, inferSchema=True)
panel    = spark.read.csv(f"{DATA}/panel.csv",        header=True, inferSchema=True)

print("exposure rows:", exposure.count())
print("conversion rows:", conv.count())
print("panel rows:", panel.count())
exposure.show(3, truncate=False)

## 2. Link (the naive attempt)

**The Why.** This is the step that separates analysts. The `household_id` key
arrives in inconsistent surface forms across sources (`HH000123`, `hh000123`,
`  HH000123 `). A join on the raw key looks like it works and silently drops
every row whose case or whitespace doesn't match. The output below quantifies
exactly how many exposure rows vanish. Nobody warns you; the join just returns
fewer rows.

**The How.** Inner-join exposure to panel on the raw key, then compare matched
rows to total exposure rows. The gap is the damage.

In [ ]:
naive = panel.join(exposure, on="household_id", how="inner")
matched, total = naive.count(), exposure.count()
print(f"naive inner join matched {matched} of {total} exposure rows")
print(f"  -> {total - matched} rows silently lost to key drift")

## 3. Shape

**The Why.** This is 70-80% of real analyst time and never makes the slide.
Three defects to neutralize: (1) key drift, fixed by normalizing every key to
`UPPER(TRIM(...))`; (2) double-logged impressions, removed with `dropDuplicates`;
(3) nulls, where blank devices become `Unknown` and blank panel segments become
`Unclassified`. Note the null trap: empty CSV fields read as true `NULL`, so a
`== ""` test misses them. You must test `isNull()` explicitly.

**The How.** A reusable `clean_key` function applied to all three frames, then
targeted dedup and null handling.

In [ ]:
def clean_key(df):
    return df.withColumn("household_id", F.upper(F.trim(F.col("household_id"))))

exposure_c, conv_c, panel_c = clean_key(exposure), clean_key(conv), clean_key(panel)

before = exposure_c.count()
exposure_c = exposure_c.dropDuplicates()
print(f"exposure dedup: {before} -> {exposure_c.count()} rows")

exposure_c = exposure_c.withColumn(
    "primary_device",
    F.when(F.col("primary_device").isNull() | (F.trim(F.col("primary_device")) == ""),
           "Unknown").otherwise(F.col("primary_device")))

panel_c = panel_c.withColumn(
    "segment",
    F.when(F.col("segment").isNull() | (F.trim(F.col("segment")) == ""),
           "Unclassified").otherwise(F.col("segment")))
print("shape complete")

## 4. Link (correct)

**The Why.** With clean keys, the join recovers the lost rows. Two design choices
carry real meaning: the **panel is the spine** (left base), so households with
zero exposures survive instead of disappearing, which prevents survivorship bias
in the conversion rate. And conversions collapse to a distinct flag before
joining, so a household with two purchase events can't double-count itself.

**The How.** Panel `LEFT JOIN` cleaned exposure (fill missing frequency with 0),
then `LEFT JOIN` a distinct conversion flag (fill non-converters with 0).

In [ ]:
master = (panel_c
          .join(exposure_c.select("household_id", "exposure_frequency", "primary_device"),
                on="household_id", how="left")
          .fillna({"exposure_frequency": 0, "primary_device": "Unknown"}))

conv_flag = conv_c.select("household_id").distinct().withColumn("converted", F.lit(1))
master = master.join(conv_flag, on="household_id", how="left").fillna({"converted": 0})

print("master households:", master.count(), "| converted:", master.filter("converted = 1").count())
master.show(3, truncate=False)

## 5. Reduce

**The Why.** A million household rows answer no question until collapsed into
something a media planner can act on. Two reductions: conversion rate **weighted
by panel weight** (so a representative segment isn't outvoted by an oversampled
one), ranked with a **window function**; and the conversion-vs-frequency curve
that exposes saturation, the point past which more impressions stop paying.

**The How.** `groupBy().agg()` for the weighted rate, a `Window` to rank, and a
second aggregation over exposure frequency.

In [ ]:
seg = (master.groupBy("segment").agg(
        F.count("*").alias("households"),
        F.round(F.avg("exposure_frequency"), 2).alias("avg_freq"),
        F.sum(F.col("converted") * F.col("panel_weight")).alias("w_conv"),
        F.sum("panel_weight").alias("w_total"))
      .withColumn("weighted_cvr", F.round(F.col("w_conv") / F.col("w_total"), 4)))

rank_w = Window.orderBy(F.col("weighted_cvr").desc())
seg = seg.withColumn("rank", F.row_number().over(rank_w)) \
         .select("rank", "segment", "households", "avg_freq", "weighted_cvr")
seg.show(truncate=False)

freq = (master.groupBy("exposure_frequency")
        .agg(F.count("*").alias("hh"), F.round(F.avg("converted"), 4).alias("cvr"))
        .orderBy("exposure_frequency"))
freq.show(8, truncate=False)

## 6. Infer

**The Why.** The segment table shows *what* converts; a model estimates *what
drives* conversion holding the other factors constant. A weighted logistic
regression returns one coefficient per factor: positive means it lifts the odds
of converting. Using `weightCol=panel_weight` makes the estimates
population-representative rather than panel-shaped, a step that is easy to
skip and costly to have skipped. `RFormula` expands the categorical fields into the design matrix
so the coefficients map cleanly back to named features.

**The How.** `RFormula` builds the features, `LogisticRegression` fits with panel
weights, then we sort coefficients by absolute size to read the drivers.

*Honesty note:* AUC lands around 0.68 by design. Conversion here is genuinely
noisy (as it is in life), so the model ranks drivers credibly without pretending
to a precision the data can't support. The recovered order, high-HHI cordcutters
and streaming-heavy viewers on top, frequency positive but secondary, matches the
true generating process.

In [ ]:
from pyspark.ml.feature import RFormula
from pyspark.ml.classification import LogisticRegression

model_df = master.filter(F.col("segment") != "Unclassified")

rf = RFormula(formula=MODEL_FORMULA, featuresCol="features", labelCol="label")
rf_model = rf.fit(model_df)
trans = rf_model.transform(model_df)

lr = LogisticRegression(featuresCol="features", labelCol="label",
                        weightCol=WEIGHT_COL, maxIter=50)
lr_model = lr.fit(trans)

# Map expanded feature names back to coefficients via the vector's metadata
meta = trans.schema["features"].metadata["ml_attr"]
names = [None] * meta["num_attrs"]
for group in meta["attrs"].values():
    for a in group:
        names[a["idx"]] = a["name"]

drivers = sorted(zip(names, lr_model.coefficients), key=lambda x: abs(x[1]), reverse=True)
print(f"AUC: {lr_model.summary.areaUnderROC:.3f} | intercept: {lr_model.intercept:.3f}\n")
print("Conversion drivers (by |coefficient|):")
for name, coef in drivers:
    print(f"  {coef:+.4f}  {name}")

## 7. Translate

**The Why.** An insight nobody can see is an insight that doesn't exist. The final
step moves the small aggregated frames into pandas (safe, because they're tiny
after reduction) and renders two pictures: the segment ranking and the saturation
curve. This is the deliverable a
non-technical reader actually looks at.

**The How.** `toPandas()` on the reduced frames, then a two-panel matplotlib chart
saved to disk.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # in Databricks you can drop this line and display inline
import matplotlib.pyplot as plt

seg_pd, freq_pd = seg.toPandas(), freq.toPandas()

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].barh(seg_pd["segment"][::-1], seg_pd["weighted_cvr"][::-1], color="#2b6cb0")
ax[0].set_title("Weighted conversion rate by segment"); ax[0].set_xlabel("weighted CVR")
ax[1].plot(freq_pd["exposure_frequency"], freq_pd["cvr"], marker="o", color="#dd6b20")
ax[1].set_title("Conversion vs exposure frequency (saturation)")
ax[1].set_xlabel("exposure frequency"); ax[1].set_ylabel("CVR")
plt.tight_layout()
plt.savefig(f"{DATA}/drivers_chart.png", dpi=110)
plt.show()
print("chart written to", f"{DATA}/drivers_chart.png")

## Why this is a repeatable process, not a one-off

Every lever lives in the Config block: re-point `DATA` at next quarter's files,
swap `CAMPAIGN_ID`, or extend `MODEL_FORMULA` with a new factor, and the same six
operations rerun unchanged. That is the difference between answering a question
once and building the workbook that answers it every quarter.